<div dir="rtl">
<h1>مرکز و پراکندگی کدام بردار؟</h1>
<p>درس 45 از 76 · نرمال‌سازی روی کدام محور انجام می‌شود؟ · <code dir="ltr">39-layernorm</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/39-layernorm.html">📖 بازگشت به همین درس</a></p>
<p>Layer Normalization را روی محور درست و با Gamma/Beta واقعی بازسازی کنید.</p><p>پیش‌نیاز: میانگین، واریانس با correction=0 و Broadcasting را مرور کنید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>بردارهای [1,2,3] و [101,102,103] پس از تنظیم مرکز چه نسبتی دارند؟ بردار ثابت بدون Epsilon چه خطری دارد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch import nn
x = torch.tensor([[[1.,2.,3.],[101.,102.,103.]],[[5.,5.,5.],[-1.,0.,4.]]])
layer = nn.LayerNorm(3,eps=1e-5)
with torch.no_grad():
    layer.weight.copy_(torch.tensor([1.,2.,0.5]))
    layer.bias.copy_(torch.tensor([0.1,-0.2,0.3]))
print('input:',x)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع manual_norm(x, gamma, beta, eps) را بدون nn.LayerNorm یا F.layer_norm بنویسید. آمار فقط روی محور آخر باشد؛ Shape و محورهای Batch/Time حفظ شوند.</p>
</div>

In [ ]:
def manual_norm(x, gamma, beta, eps):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = manual_norm(x,layer.weight,layer.bias,layer.eps)
    if result is None: return False
    torch.testing.assert_close(result,layer(x))
    torch.testing.assert_close(result[0,0],result[0,1])
    torch.testing.assert_close(result[1,0],layer.bias)
    other = torch.arange(20.).reshape(1,4,5)
    reference = nn.LayerNorm(5)
    torch.testing.assert_close(manual_norm(other,reference.weight,reference.bias,reference.eps),reference(other))
    assert torch.isfinite(result).all()
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط به تمام ویژگی‌های موقعیت آخر ۱۰۰ اضافه کنید. با ثابت‌بودن Gamma/Beta، خروجی آن موقعیت تقریباً ثابت می‌ماند؛ بقیهٔ موقعیت‌ها نیز نباید اثر بگیرند.</p>
</div>

In [ ]:
changed = x.clone(); changed[:,-1] += 100
with torch.no_grad():
    print('per-position normalized difference:',(layer(changed)-layer(x)).abs().amax(-1))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>نسخهٔ خراب آمار را روی محور زمان می‌گیرد و موقعیت‌ها را به هم وابسته می‌کند. تابع normalize_features(x,eps) را با Gamma=1 و Beta=0 اصلاح کنید.</p>
</div>

In [ ]:
wrong = (x-x.mean(1,keepdim=True))/torch.sqrt(x.var(1,correction=0,keepdim=True)+1e-5)
print('wrong time-axis normalization:',wrong)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def normalize_features(x, eps):
    # TODO
    return None

In [ ]:
def test_repair():
    result = normalize_features(x,1e-5)
    if result is None: return False
    torch.testing.assert_close(result,nn.LayerNorm(3,elementwise_affine=False)(x))
    constant = torch.full((1,1,3),5.)
    torch.testing.assert_close(normalize_features(constant,1e-5),torch.zeros_like(constant))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>norm_1، norm_2 و final_norm در مدل همین قرارداد آخرین محور را دارند. مخلوط‌کردن زمان در محاسبهٔ آمار می‌تواند حتی با Attention پوشیده، اطلاعات آینده را وارد گذشته کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>پس از Gamma و Beta دلخواه، چرا دیگر نباید برای هر خروجی میانگین دقیقاً صفر و واریانس دقیقاً یک assert کنیم؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/39-layernorm.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/39-layernorm.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>